# Extraccion de features: notas de exploracion de los datos para la selección de features significativas.

## WINDOWS

### Events
Provider, level, channel y computer tienen siempre los mismos valores. Así que los elementos que hay que convertir en features utiles son EventID y el timestamp. El resultado será la entropia de las frecuencias de los identificadores de los eventos y el grupo temporal en el que estan (para poder fusionarlo bien con el resto de logs de la muestra). Tiene tendencialmente una entropia más alta los eventos de las muestras infectadas.



In [186]:
import os 
import pandas as pd
from scipy.stats import entropy

## Extraccion de features de eventos de windows

def extract_events_features_windows(df_events):
    df_events = df_events[['EventID', 'TimeCreated']].sort_values(by='TimeCreated', ascending=True)
    temporal_window = []
    ventana = pd.Timedelta(seconds=30)
    ventana_inicio = df_events['TimeCreated'].min()
    df_result = pd.DataFrame(columns=['EventID_entropy', 'Event_count'])
    
    
    ## Creamos la variable de ventana temporal
    for index, row in df_events.iterrows():
        temporal_window.append(int(((row['TimeCreated'] - ventana_inicio) / ventana)))
    df_events['temporal_window'] = temporal_window
    
    # Agrupamos por ventana temporal y contamos la entropia de cada EventID
    index = df_events['temporal_window'].max()
    temporal_windows = []
    eventid_entropies = []
    event_count = []
    for i in range (0, index+1):
        temporal_windows.append(i)
        serie_events = df_events[df_events['temporal_window'] == i]['EventID'].value_counts()
        eventid_entropies.append(entropy(serie_events, base=2))
        event_count.append(df_events[df_events['temporal_window'] == i]['EventID'].size)
    
    df_result['EventID_entropy'] = eventid_entropies
    df_result.index = temporal_windows
    df_result.index.name = 'temporal_window'
    df_result['Event_count'] = event_count
    return df_result




### Filesystem events
Las features escogidas para este log son las siguientes para cada ventana de 30 segundos: numero total de eventos, numero de eventos de creacion, numero de eventos de modificacion, numero de eliminaciones, media de longitud de los paths de origen, numero de paths que aparecen solo una vez y porcentaje de eventos que son de directorios en vez de sobre archivos.

In [187]:
# Extraccion de features de filesystem de windows

def extract_filesystem_features_windows(df_filesystem):
    timestamp = pd.to_datetime(df_filesystem['Date'] + ' ' + df_filesystem['Time'], format='%Y-%m-%d %H:%M:%S')
    df_filesystem.pop('Date')
    df_filesystem.pop('Time')
    df_filesystem['timestamp'] = timestamp
    df_result = pd.DataFrame(columns=['Total_events', 'Create_events', 'Modify_events', 'Delete_events', 'Avg_path_length', 'Unique_path_count', 'Directory_event_ratio'])
    temporal_window = []
    ventana = pd.Timedelta(seconds=30)
    ventana_inicio = df_filesystem['timestamp'].min()
    for index, row in df_filesystem.iterrows():
        temporal_window.append(int(((row['timestamp'] - ventana_inicio) / ventana)))
    df_filesystem['temporal_window'] = temporal_window
    index = df_filesystem['temporal_window'].max()
    total_events = []
    create_events = []
    modify_events = []
    delete_events = []
    avg_path_length = []
    unique_path_count = []
    directory_event_ratio = []
    temporal_windows = []
    
    for i in range (0, index+1):
        temporal_windows.append(i)
        total_events.append(df_filesystem[df_filesystem['temporal_window'] == i]['Event'].count())
        create_events.append(df_filesystem[(df_filesystem['temporal_window'] == i) & (df_filesystem['Event'] == 'CREATED')]['Event'].count())
        modify_events.append(df_filesystem[(df_filesystem['temporal_window'] == i) & (df_filesystem['Event'] == 'MODIFIED')]['Event'].count())
        delete_events.append(df_filesystem[(df_filesystem['temporal_window'] == i) & (df_filesystem['Event'] == 'DELETED')]['Event'].count())
        avg_path_length.append(df_filesystem[(df_filesystem['temporal_window'] == i)]['Path src'].apply(lambda x: len(x.split('\\'))).mean())
        unique_path_count.append(df_filesystem[(df_filesystem['temporal_window'] == i)]['Path src'].nunique())
        directory_event_ratio.append(len(df_filesystem[(df_filesystem['temporal_window'] == i) & (df_filesystem['isDirectory'] == True)]['Event']) / (len(df_filesystem[(df_filesystem['temporal_window'] == i)]['Event'])+1))
        
 
    df_result['Total_events'] = total_events
    df_result['Create_events'] = create_events
    df_result['Modify_events'] = modify_events
    df_result['Delete_events'] = delete_events
    df_result['Avg_path_length'] = avg_path_length
    df_result['Unique_path_count'] = unique_path_count
    df_result['Directory_event_ratio'] = directory_event_ratio
    df_result.index = temporal_windows
    df_result.index.name = 'temporal_window'
    return df_result

### HW resources.

Respecto a los datos de uso de HW, se van a extraer 12 features. Vamos a comenzar con las relacionadas con la CPU:

1. Uso medio de la CPU total.
2. Desviacion típica del uso de la CPU total.
3. Ratio tiempo CPU-User / tiempo CPU total.
4. Ratio tiempo CPU-system / tiempo CPU total.
5. Ratio tiempo CPU-interrupted / tiempo CPU total.
6. Ratio tiempo CPU-waiting / tiempo CPU total.

En el caso de windows, dado los datos de CPU recolectados, tenemos las siguientes equivalencias:

- CPU-User = CPU_USER
- CPU-Interrupt = CPU_INTERRUPT + CPU_DCP
- CPU-SYSTEM = CPU_SYSTEM
- CPU-WAITING = CPU_IDLE

Respecto a los datos de uso de memoria, se van a extraer los siguientes features

7. Media de espacio de memoria usado.
8. Desviación estándar de memoria usada.
9. Ratio de memoria usada / memoria total.

Respecto a la memoria SWAP, no se va a extraer ningun feature de ella por mantener la homogeneidad. HA habido un problema de configuración con las MVs de Ubuntu y el uso de SWAP es cero aunque tenga espacio en el disco asignado para ello.

Por último, estos son los features relacionados con el uso del disco:

10. Número de lecturas / 30 segundos.
11. Número escrituras / 30 segudos.
12. Ratio escrituras / lecturas + 1 (Evita divisiones entre 0).


In [188]:
def extract_hardware_features_windows(df_hardware):
    df_hardware = df_hardware[['Timestamp', 'CPU Total (%)', 'CPU User (%)', 'CPU System (%)', 'CPU Idle (%)', 'CPU Interrupt (%)', 
                               'CPU DCP (%)', 'Mem used', 'Mem Available', 'Mem Total', 'Disco - Lecturas Completadas', 'Disco - Escrituras Completadas']].sort_values(by='Timestamp', ascending=True)
    ventana = pd.Timedelta(seconds=30)
    ventana_inicio = df_hardware['Timestamp'].min()
    df_result = pd.DataFrame(columns=['CPU_mean', 'CPU_std', 'CPU_user', 'CPU_system', 'CPU_interrupt', 'CPU_wait', 
                                      'Mem_used_mean', 'Mem_used_std', 'Used_mem_ratio', 'Disk_read', 'Disk_write', 'Disk_read_write_ratio'])
     ## Creamos la variable de ventana temporal
    temporal_window = []
    for index, row in df_hardware.iterrows():
        temporal_window.append(int(((row['Timestamp'] - ventana_inicio) / ventana)))
    df_hardware['temporal_window'] = temporal_window
    index = df_hardware['temporal_window'].max()
    mem_total = df_hardware['Mem Total'].loc[0]
    cpu_mean = []
    cpu_std = []
    cpu_user = []
    cpu_system = []
    cpu_interrupt = []
    cpu_wait = []
    mem_used_mean = []
    mem_used_std = []
    used_mem_ratio = []
    disk_read = []
    disk_write = []
    disk_read_write_ratio = []
    temporal_windows = []
    for i in range (0, index+1):
        temporal_windows.append(i)
        cpu_mean.append(df_hardware[df_hardware['temporal_window'] == i]['CPU Total (%)'].mean())
        cpu_std.append(df_hardware[df_hardware['temporal_window'] == i]['CPU Total (%)'].std())
        cpu_user.append(df_hardware[df_hardware['temporal_window'] == i]['CPU User (%)'].mean())
        cpu_system.append(df_hardware[df_hardware['temporal_window'] == i]['CPU System (%)'].mean())
        cpu_interrupt.append(df_hardware[df_hardware['temporal_window'] == i]['CPU Interrupt (%)'].mean())
        cpu_wait.append(df_hardware[df_hardware['temporal_window'] == i]['CPU DCP (%)'].mean())
        mem_used_mean.append(df_hardware[df_hardware['temporal_window'] == i]['Mem used'].mean())
        mem_used_std.append(df_hardware[df_hardware['temporal_window'] == i]['Mem used'].std())
        used_mem_ratio.append(df_hardware[df_hardware['temporal_window'] == i]['Mem used'].mean() / mem_total)
        disk_read_init = df_hardware[(df_hardware['temporal_window'] == i) & (df_hardware['Timestamp'] == df_hardware[df_hardware['temporal_window'] == i]['Timestamp'].min())]['Disco - Lecturas Completadas'].values[0]
        disk_read_end = df_hardware[(df_hardware['temporal_window'] == i) & (df_hardware['Timestamp'] == df_hardware[df_hardware['temporal_window'] == i]['Timestamp'].max())]['Disco - Lecturas Completadas'].values[0]
        disk_read.append(disk_read_end - disk_read_init)
        disk_write_init = df_hardware[(df_hardware['temporal_window'] == i) & (df_hardware['Timestamp'] == df_hardware[df_hardware['temporal_window'] == i]['Timestamp'].min())]['Disco - Escrituras Completadas'].values[0]
        disk_write_end = df_hardware[(df_hardware['temporal_window'] == i) & (df_hardware['Timestamp'] == df_hardware[df_hardware['temporal_window'] == i]['Timestamp'].max())]['Disco - Escrituras Completadas'].values[0]
        disk_write.append(disk_write_end - disk_write_init)
        disk_read_write_ratio.append((disk_write_end - disk_write_init) / ((disk_read_end - disk_read_init) + 1))
        
    df_result['CPU_mean'] = cpu_mean
    df_result['CPU_std'] = cpu_std
    df_result['CPU_user'] = cpu_user
    df_result['CPU_system'] = cpu_system
    df_result['CPU_interrupt'] = cpu_interrupt
    df_result['CPU_wait'] = cpu_wait
    df_result['Mem_used_mean'] = mem_used_mean
    df_result['Mem_used_std'] = mem_used_std
    df_result['Used_mem_ratio'] = used_mem_ratio
    df_result['Disk_read'] = disk_read
    df_result['Disk_write'] = disk_write
    df_result['Disk_read_write_ratio'] = disk_read_write_ratio
    df_result.index = temporal_windows
    df_result.index.name = 'temporal_window'
    return df_result    
    

## Servicios

Las features a extraer relacionadas con los logs de servicios de windows son las siguientes:

- Num totales de servicios
- Num servicios unicos
- Num de rutas unicas
- Num servicios auto-start

In [189]:
def extract_services_features_windows(df_services):
    df_services = df_services[['Timestamp','Nombre','Estado','Auto-Start','PID','Ruta del ejecutable','Usuario']].sort_values(by='Timestamp', ascending=True)
    ventana = pd.Timedelta(seconds=30)
    ventana_inicio = df_services['Timestamp'].min()
    df_result = pd.DataFrame(columns=['num_services', 'num_unique_paths'])
    df_services = df_services.drop_duplicates(subset=['Nombre', 'Ruta del ejecutable'])
    ## Creamos la variable de ventana temporal
    temporal_window = []
    for index, row in df_services.iterrows():
        temporal_window.append(int(((row['Timestamp'] - ventana_inicio) / ventana)))
    df_services['temporal_window'] = temporal_window
    index = df_services['temporal_window'].max()
    num_services = []
    num_unique_services = []
    num_unique_paths = []
    num_auto_start_services = []
    temporal_windows = []
    for i in range (0, index+1):
        temporal_windows.append(i)
        num_services.append(len(df_services[df_services['temporal_window'] == i]['Nombre']))
        num_unique_paths.append(df_services[df_services['temporal_window'] == i]['Ruta del ejecutable'].nunique())
    df_result['num_services'] = num_services
    df_result['num_unique_paths'] = num_unique_paths
    df_result.index = temporal_windows
    df_result.index.name = 'temporal_window'
    return df_result
        
        
        

## LINUX

### Events
Para hacer el log homogéneo con los events de windows, vamos a sacar solamente la entropia de la distribucion de keys.

In [190]:
def extract_events_features_linux(df_events):
    df_events = df_events[['key', 'timestamp']].sort_values(by='timestamp', ascending=True)
    temporal_window = []
    ventana = pd.Timedelta(seconds=30)
    df_events['timestamp'] = pd.to_datetime(df_events['timestamp'], format = 'ISO8601')
    ventana_inicio = df_events['timestamp'].min()
    df_result = pd.DataFrame(columns=['EventID_entropy', 'Event_count'])
    
     ## Creamos la variable de ventana temporal
    for index, row in df_events.iterrows():
        temporal_window.append(int(((row['timestamp'] - ventana_inicio) / ventana)))
    df_events['temporal_window'] = temporal_window
    
     # Agrupamos por ventana temporal y contamos la entropia de cada EventID
    index = df_events['temporal_window'].max()
    
    temporal_windows = []
    eventid_entropies = []
    event_count = []
    for i in range (0, index+1):
        temporal_windows.append(i)
        serie_events = df_events[df_events['temporal_window'] == i]['key'].value_counts()
        eventid_entropies.append(entropy(serie_events, base=2))
        event_count.append(df_events[df_events['temporal_window'] == i]['key'].size)
    
    df_result['EventID_entropy'] = eventid_entropies
    df_result.index = temporal_windows
    df_result.index.name = 'temporal_window'
    df_result['Event_count'] = event_count
    
    return df_result
    




### Filesystem events
Para extraer las fetures relevantes de este log, se sigue el mismo proeceso que en windows, ya que el conjunto de datos es el mismo. Este es el numero total de eventos, numero de eventos de creacion, numero de eventos de modificacion, numero de eliminaciones, media de longitud de los paths de origen, numero de paths que aparecen solo una vez y porcentaje de eventos que son de directorios en vez de sobre archivos.

In [ ]:
# Extraccion de features de filesystem de linux

def extract_filesystem_features_linux(df_filesystem):
    timestamp = pd.to_datetime(df_filesystem['Date'] + ' ' + df_filesystem['Time'], format='%Y-%m-%d %H:%M:%S')
    df_filesystem.pop('Date')
    df_filesystem.pop('Time')
    df_filesystem['timestamp'] = timestamp
    df_result = pd.DataFrame(columns=['Total_events', 'Create_events', 'Modify_events', 'Delete_events', 'Avg_path_length', 'Unique_path_count', 'Directory_event_ratio'])
    temporal_window = []
    ventana = pd.Timedelta(seconds=30)
    ventana_inicio = df_filesystem['timestamp'].min()
    for index, row in df_filesystem.iterrows():
        temporal_window.append(int(((row['timestamp'] - ventana_inicio) / ventana)))
    df_filesystem['temporal_window'] = temporal_window
    index = df_filesystem['temporal_window'].max()
    total_events = []
    create_events = []
    modify_events = []
    delete_events = []
    avg_path_length = []
    unique_path_count = []
    directory_event_ratio = []
    temporal_windows = []
    
    for i in range (0, index+1):
        temporal_windows.append(i)
        total_events.append(df_filesystem[df_filesystem['temporal_window'] == i]['Event'].count())
        create_events.append(df_filesystem[(df_filesystem['temporal_window'] == i) & (df_filesystem['Event'] == 'CREATED')]['Event'].count())
        modify_events.append(df_filesystem[(df_filesystem['temporal_window'] == i) & (df_filesystem['Event'] == 'MODIFIED')]['Event'].count())
        delete_events.append(df_filesystem[(df_filesystem['temporal_window'] == i) & (df_filesystem['Event'] == 'DELETED')]['Event'].count())
        avg_path_length.append(df_filesystem[(df_filesystem['temporal_window'] == i)]['Path src'].apply(lambda x: len(x.split('/'))).mean())
        unique_path_count.append(df_filesystem[(df_filesystem['temporal_window'] == i)]['Path src'].nunique())
        directory_event_ratio.append(len(df_filesystem[(df_filesystem['temporal_window'] == i) & (df_filesystem['isDirectory'] == True)]['Event']) / (len(df_filesystem[(df_filesystem['temporal_window'] == i)]['Event'])+1))
    df_result['Total_events'] = total_events
    df_result['Create_events'] = create_events
    df_result['Modify_events'] = modify_events
    df_result['Delete_events'] = delete_events
    df_result['Avg_path_length'] = avg_path_length
    df_result['Unique_path_count'] = unique_path_count
    df_result['Directory_event_ratio'] = directory_event_ratio
    df_result.index = temporal_windows
    df_result.index.name = 'temporal_window'
    return df_result

### Hardware resources

Respecto a los datos de uso de HW, se van a extraer 12 features. Vamos a comenzar con las relacionadas con la CPU:

1. Uso medio de la CPU total.
2. Desviacion típica del uso de la CPU total.
3. Ratio tiempo CPU-User / tiempo CPU total.
4. Ratio tiempo CPU-system / tiempo CPU total.
5. Ratio tiempo CPU-interrupted / tiempo CPU total.
6. Ratio tiempo CPU-waiting / tiempo CPU total.

En el caso de windows, dado los datos de CPU recolectados, tenemos las siguientes equivalencias:

- CPU-User = CPU_guest_nice + CPU_nice + CPU_user + CPU_guest
- CPU-Interrupt = CPU_irq + CPU_soft_irq
- CPU-SYSTEM = CPU_SYSTEM
- CPU-WAITING = CPU_IDLE + CPU_IOWait + CPU_steal

Respecto a los datos de uso de memoria, se van a extraer los siguientes features

7. Media de espacio de memoria usado.
8. Desviación estándar de memoria usada.
9. Ratio de memoria usada / memoria total.

Respecto a la memoria SWAP, no se va a extraer ningun feature de ella por mantener la homogeneidad. Ha habido un problema de configuración con las MVs de Ubuntu y el uso de SWAP es cero aunque tenga espacio en el disco asignado para ello.

Por último, estos son los features relacionados con el uso del disco:

10. Número de lecturas / 30 segundos.
11. Número escrituras / 30 segudos.
12. Ratio escrituras / lecturas + 1 (Evita divisiones entre 0).

In [192]:
def extract_hardware_features_linux(df_hardware):
    df_hardware = df_hardware[['Timestamp','CPU Total','CPU User','CPU Nice','CPU System','CPU Idle','CPU Iowait','CPU Irq',
                               'CPU SoftIrq','CPU Steal','CPU Guest','CPU Guest nice','Mem Total','Mem Available','Mem Percent',
                               'Mem used','Mem Free','Mem Active','Mem Inactive','Buffers','Cached','Shared','Slab','Swap Total',
                               'Swap Used','Swap Free','Swap Percent','Swap Sin','Swap Sout','Disco - Lecturas Completadas',
                               'Disco - Escrituras Completadas']].sort_values(by='Timestamp', ascending=True)
    ventana = pd.Timedelta(seconds=30)
    ventana_inicio = df_hardware['Timestamp'].min()
    df_result = pd.DataFrame(columns=['CPU_mean', 'CPU_std', 'CPU_user', 'CPU_system', 'CPU_interrupt', 'CPU_wait', 
                                      'Mem_used_mean', 'Mem_used_std', 'Used_mem_ratio', 'Disk_read', 'Disk_write', 'Disk_read_write_ratio'])
     ## Creamos la variable de ventana temporal
    temporal_window = []
    for index, row in df_hardware.iterrows():
        temporal_window.append(int(((row['Timestamp'] - ventana_inicio) / ventana)))
    df_hardware['temporal_window'] = temporal_window
    index = df_hardware['temporal_window'].max()
    mem_total = df_hardware['Mem Total'].loc[0]
    cpu_mean = []
    cpu_std = []
    cpu_user = []
    cpu_system = []
    cpu_interrupt = []
    cpu_wait = []
    mem_used_mean = []
    mem_used_std = []
    used_mem_ratio = []
    disk_read = []
    disk_write = []
    disk_read_write_ratio = []
    temporal_windows = []
    for i in range (0, index+1):
        temporal_windows.append(i)
        cpu_mean.append(df_hardware[df_hardware['temporal_window'] == i]['CPU Total'].mean())
        cpu_std.append(df_hardware[df_hardware['temporal_window'] == i]['CPU Total'].std())
        cpu_user.append(df_hardware[df_hardware['temporal_window'] == i]['CPU User'].mean() + 
                        df_hardware[df_hardware['temporal_window'] == i]['CPU Nice'].mean() + 
                        df_hardware[df_hardware['temporal_window'] == i]['CPU Guest'].mean() +
                        df_hardware[df_hardware['temporal_window'] == i]['CPU Guest nice'].mean())
        cpu_system.append(df_hardware[df_hardware['temporal_window'] == i]['CPU System'].mean())
        cpu_interrupt.append(df_hardware[df_hardware['temporal_window'] == i]['CPU Irq'].mean() + 
                            df_hardware[df_hardware['temporal_window'] == i]['CPU SoftIrq'].mean())
        cpu_wait.append(df_hardware[df_hardware['temporal_window'] == i]['CPU Idle'].mean() +
                        df_hardware[df_hardware['temporal_window'] == i]['CPU Iowait'].mean() +
                        df_hardware[df_hardware['temporal_window'] == i]['CPU Steal'].mean())
        
        mem_used_mean.append(df_hardware[df_hardware['temporal_window'] == i]['Mem used'].mean())
        mem_used_std.append(df_hardware[df_hardware['temporal_window'] == i]['Mem used'].std())
        used_mem_ratio.append(df_hardware[df_hardware['temporal_window'] == i]['Mem used'].mean() / mem_total)
        disk_read_init = df_hardware[(df_hardware['temporal_window'] == i) & (df_hardware['Timestamp'] == df_hardware[df_hardware['temporal_window'] == i]['Timestamp'].min())]['Disco - Lecturas Completadas'].values[0]
        disk_read_end = df_hardware[(df_hardware['temporal_window'] == i) & (df_hardware['Timestamp'] == df_hardware[df_hardware['temporal_window'] == i]['Timestamp'].max())]['Disco - Lecturas Completadas'].values[0]
        disk_read.append(disk_read_end - disk_read_init)
        disk_write_init = df_hardware[(df_hardware['temporal_window'] == i) & (df_hardware['Timestamp'] == df_hardware[df_hardware['temporal_window'] == i]['Timestamp'].min())]['Disco - Escrituras Completadas'].values[0]
        disk_write_end = df_hardware[(df_hardware['temporal_window'] == i) & (df_hardware['Timestamp'] == df_hardware[df_hardware['temporal_window'] == i]['Timestamp'].max())]['Disco - Escrituras Completadas'].values[0]
        disk_write.append(disk_write_end - disk_write_init)
        disk_read_write_ratio.append((disk_write_end - disk_write_init) / ((disk_read_end - disk_read_init) + 1))
        
    df_result['CPU_mean'] = cpu_mean
    df_result['CPU_std'] = cpu_std
    df_result['CPU_user'] = cpu_user
    df_result['CPU_system'] = cpu_system
    df_result['CPU_interrupt'] = cpu_interrupt
    df_result['CPU_wait'] = cpu_wait
    df_result['Mem_used_mean'] = mem_used_mean
    df_result['Mem_used_std'] = mem_used_std
    df_result['Used_mem_ratio'] = used_mem_ratio
    df_result['Disk_read'] = disk_read
    df_result['Disk_write'] = disk_write
    df_result['Disk_read_write_ratio'] = disk_read_write_ratio
    df_result.index = temporal_windows
    df_result.index.name = 'temporal_window'
    return df_result    


## Features de servicios

Los features a extraer son los mismos que en caso de windows. 

In [193]:
def extract_services_features_linux(df_services):
    df_services = df_services[['timestamp','service_name','load_state','active_state','sub_state','description','pid','memory_usage','cpu_usage',
                               'file_path','time_start','time_end']].sort_values(by='timestamp', ascending=True)
    ventana = pd.Timedelta(seconds=30)
    ventana_inicio = df_services['timestamp'].min()
    df_result = pd.DataFrame(columns=['num_services', 'num_unique_paths'])
    df_services = df_services.drop_duplicates(subset=['service_name', 'file_path'])
    ## Creamos la variable de ventana temporal
    temporal_window = []
    for index, row in df_services.iterrows():
        temporal_window.append(int(((row['timestamp'] - ventana_inicio) / ventana)))
    df_services['temporal_window'] = temporal_window
    index = df_services['temporal_window'].max()
    num_services = []

    num_unique_paths = []
    
    temporal_windows = []
    for i in range (0, index+1):
        temporal_windows.append(i)
        num_services.append(len(df_services[df_services['temporal_window'] == i]['service_name']))
        num_unique_paths.append(df_services[df_services['temporal_window'] == i]['file_path'].nunique())
    df_result['num_services'] = num_services

    df_result['num_unique_paths'] = num_unique_paths

    df_result.index = temporal_windows
    df_result.index.name = 'temporal_window'
    return df_result

# Ambos SO

## Procesos

Dada la similitud de los logs generados de procesos en ambos SO, se puede usar el mismo script para extraer las features correspondientes. Los logs consisten en una captura, cada 5 segundos, de todos los procesos activos e información sobre ellos. Esto hace que se incorpore el paso intermedio de eliminar los duplicados. Para ello se considera duplicado todo par de procesos con mismo PID y tiempo de creación. Las features que se van a extraer son las siguientes:
- Número de procesos
- Media de lecturas
- Media de escrituras
- Media bytes leidos
- Media bytes escritos
- Media bytes escritos / media bytes leidos +1
- STD bytes escritos
- Porcentaje de procesos con más escrituras que lecturas
- Entropía del path donde se ubica el archivo del proceso
- Número de procesos sin path válidos
- Numero de procesos huerfanos
- Numero de padres distintos / numero de procesos
- Numero de usuarios unicos

In [194]:
def extract_features_processes(df_processes):
    df_processes = df_processes[['Timestamp', 'PID', 'Nombre', 'Ruta', 'Usuario','Tiempo de creación', 'Proceso padre',
                                 'Numero lecturas', 'Bytes leidos', 'Numero escrituras', 'Bytes escritos']].sort_values(by='Timestamp', ascending=True)
    df_processes = df_processes.drop_duplicates(subset=['PID', 'Tiempo de creación'])
    ventana = pd.Timedelta(seconds=30)
    ventana_inicio = df_processes['Timestamp'].min()
    df_result = pd.DataFrame(columns=['Num_processes', 'Avg_num_reads', 'Avg_num_writes', 'Avg_bytes_read', 'Avg_bytes_written', 'read_write_ratio',
                                      'std_bytes_written', 'procceses_with_more_writes_than_reads_percentage', 'entropy_path',
                                      'processes_without_path', 'orphan_processes', 'avg_process_lifetime', 'different_parents_per_processes', 'unique_users'])
     ## Creamos la variable de ventana temporal
    temporal_window = []
    for index, row in df_processes.iterrows():
        temporal_window.append(int(((row['Timestamp'] - ventana_inicio) / ventana)))
    df_processes['temporal_window'] = temporal_window
    index = df_processes['temporal_window'].max()
    num_processes = []
    avg_num_reads = []
    avg_num_writes = []
    avg_bytes_read = [] 
    avg_bytes_written = []
    write_read_ratio = []
    std_bytes_written = []
    processes_with_more_writes_than_reads_percentage = []
    entropy_path = []
    processes_without_path = []
    orphan_processes = []
    different_parents_per_processes = []
    unique_users = []
    temporal_windows = []
    for i in range (0, index+1):
        temporal_windows.append(i)
        num_processes.append(len(df_processes[df_processes['temporal_window'] == i]))
        avg_num_reads.append(df_processes[df_processes['temporal_window'] == i]['Numero lecturas'].mean())
        avg_num_writes.append(df_processes[df_processes['temporal_window'] == i]['Numero escrituras'].mean())
        avg_bytes_read.append(df_processes[df_processes['temporal_window'] == i]['Bytes leidos'].mean())
        avg_bytes_written.append(df_processes[df_processes['temporal_window'] == i]['Bytes escritos'].mean())
        write_read_ratio.append(avg_bytes_written[-1] / (avg_bytes_read[-1] + 1))
        std_bytes_written.append(df_processes[df_processes['temporal_window'] == i]['Bytes escritos'].std())
        proc_more_reads = len(df_processes[(df_processes['temporal_window'] == i) & (df_processes['Numero escrituras'] > df_processes['Numero lecturas'])])
        processes_with_more_writes_than_reads_percentage.append((proc_more_reads / (num_processes[-1] +1)) * 100)
        paths = df_processes[df_processes['temporal_window'] == i]['Ruta'].value_counts()
        entropy_path.append(entropy(paths, base=2))  
        processes_without_path.append(len(df_processes[(df_processes['temporal_window'] == i) & (df_processes['Ruta'].isnull())]))
        orphan_processes.append(len(df_processes[(df_processes['temporal_window'] == i) & (df_processes['Proceso padre'].isnull())]))
        different_parents_per_processes.append(df_processes[df_processes['temporal_window'] == i]['Proceso padre'].value_counts().std())
        unique_users.append(df_processes[df_processes['temporal_window'] == i]['Usuario'].nunique())
    df_result['Num_processes'] = num_processes
    df_result['Avg_num_reads'] = avg_num_reads
    df_result['Avg_num_writes'] = avg_num_writes
    df_result['Avg_bytes_read'] = avg_bytes_read
    df_result['Avg_bytes_written'] = avg_bytes_written
    df_result['write_read_ratio'] = write_read_ratio
    df_result['std_bytes_written'] = std_bytes_written
    df_result['procceses_with_more_writes_than_reads_percentage'] = processes_with_more_writes_than_reads_percentage
    df_result['entropy_path'] = entropy_path
    df_result['processes_without_path'] = processes_without_path
    df_result['orphan_processes'] = orphan_processes
    df_result['different_parents_per_processes'] = different_parents_per_processes
    df_result['unique_users'] = unique_users
    df_result.index = temporal_windows
    df_result.index.name = 'temporal_window'
     
    return df_result


In [195]:
def test_win():
    data_file_path = os.path.abspath('.')
    data_path = os.path.join(data_file_path, 'datos/windows/infected/WannaCry/WannaCry2/services.csv')
    data_path1 = os.path.join(data_file_path, 'datos/windows/noninfected/datos6/services.csv')
    data = pd.read_csv(data_path, parse_dates=['Timestamp'])
    data1 = pd.read_csv(data_path1, parse_dates=['Timestamp'])
    df_infected = extract_services_features_windows(data)
    df_noninfected = extract_services_features_windows(data1)
    print("----------- infected -------------")
    print(df_infected)
    print("----------- non infected -------------")
    print(df_noninfected)


    
def test_linux():
    data_file_path = os.path.abspath('.')
    data_path = os.path.join(data_file_path, 'datos/linux/infected/fog/fog2/services.csv')
    data_path1 = os.path.join(data_file_path, 'datos/linux/noninfected/datos6/services.csv')
    data = pd.read_csv(data_path, parse_dates=['timestamp'])
    data1 = pd.read_csv(data_path1, parse_dates=['timestamp'])
    df_infected = extract_services_features_linux(data)
    df_noninfected = extract_services_features_linux(data1)
    print("----------- infected -------------")
    print(df_infected)
    print("----------- non infected -------------")
    print(df_noninfected)


test_win()
test_linux()

----------- infected -------------
                 num_services  num_unique_paths
temporal_window                                
0                         251               103
----------- non infected -------------
                 num_services  num_unique_paths
temporal_window                                
0                         251               103
----------- infected -------------
                 num_services  num_unique_paths
temporal_window                                
0                         178                 1
----------- non infected -------------
                 num_services  num_unique_paths
temporal_window                                
0                         183                 1
1                           0                 0
2                           0                 0
3                           0                 0
4                           1                 1


# Estructuración del archivo final de datos.

En esta sección se creará un dataframe final con todas las ventanas temporales de todos los datos. Los campos presentes serán los features extraidos en las anteriores secciones más dos campos adicionales: el SO de la muestra (Windows o Linux) y el campo que indica si es una muestra infectada o no de reansomware. En principio se omitirá la información de que tipo de malware tiene en caso de las muestras infectadas.


In [196]:
def join_features(path):
    path = os.path.abspath(path)
    print(os.listdir(path))
    df_events = extract_events_features_windows(pd.read_csv(os.path.join(path, 'events.csv'), parse_dates=['TimeCreated']))
    df_filesystem = extract_filesystem_features_windows(pd.read_csv(os.path.join(path, 'filesystem_event.csv')))
    df_hardware = extract_hardware_features_windows(pd.read_csv(os.path.join(path, 'HW_resources.csv'), parse_dates=['Timestamp']))
    df_services = extract_services_features_windows(pd.read_csv(os.path.join(path, 'services.csv'), parse_dates=['Timestamp']))
    df_processes = extract_features_processes(pd.read_csv(os.path.join(path, 'processes.csv'), parse_dates=['Timestamp']))
    print(df_services)
join_features('./datos/windows/noninfected/datos4/')

['events.csv', 'filesystem_event.csv', 'HW_resources.csv', 'packets.csv', 'processes.csv', 'services.csv']
                 num_services  num_unique_paths
temporal_window                                
0                         251               103
